# Bengali WH Question Generation from Facts (Qwen3.5, GPU)

Generates **one WH question per fact** (`subject` / `attribute` / `value` triples) for each `text` entry in a JSONL dataset. Since the answer to each question is already the fact's `value`, the model only has to phrase the question — the answer is taken directly from the fact, so it's grounded by construction (no hallucinated answers).

**Before running:**
- Kaggle notebook settings → Accelerator → **GPU T4 x2**
- Settings → Internet → **On** (needed to download the model from Hugging Face)
- Attach your dataset, then update `INPUT_PATH` below to point at the `.jsonl` file

**Model sizing for T4x2 (16GB each):**
| Model | VRAM (4-bit) | Notes |
|---|---|---|
| `Qwen/Qwen3.5-4B` | ~3GB | fastest, lowest quality |
| `Qwen/Qwen3.5-9B` | ~6GB | **recommended default** — fits on 1 T4 with headroom to batch |
| `Qwen/Qwen3.5-35B-A3B` (MoE) | ~18-20GB | needs both T4s via `device_map="auto"`, slower setup, better quality |


In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

In [ ]:
import json, re, os, time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("GPUs available:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  {i}: {torch.cuda.get_device_name(i)} - {props.total_memory/1e9:.1f} GB")

## Config

In [ ]:
# ---- Paths ----
# CHANGE THIS to wherever your new JSONL file lives, e.g.
# "/kaggle/input/<your-dataset>/train.jsonl"
INPUT_PATH = "/kaggle/input/<your-dataset>/facts_dataset.jsonl"
OUTPUT_PATH = "/kaggle/working/question_answer.json"
PROGRESS_PATH = "/kaggle/working/question_answer.progress.json"
CHECKPOINT_EVERY = 20     # save progress every N records processed

# ---- Model ----
# Swap this to Qwen/Qwen3.5-4B for more speed, or Qwen/Qwen3.5-35B-A3B for more quality (needs both GPUs)
MODEL_NAME = "Qwen/Qwen3.5-9B"

# ---- Generation ----
BATCH_SIZE = 8            # records processed per generate() call - raise if VRAM allows
MAX_NEW_TOKENS = 900       # bumped up vs. before since fact count per record can be large
LIMIT = None               # set an int (e.g. 20) for a quick smoke test before a full run

## Load model (4-bit quantized)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"          # required for correct batched generation on decoder-only models
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.eval()

print("Model loaded. Device map:", getattr(model, "hf_device_map", "n/a"))

## Prompt template + parsing helpers

**Changed from the previous version:** instead of asking for 8 free-form Q/A pairs from a context, we now hand the model the paragraph *and* the exact list of `subject/attribute/value` facts, and ask it for exactly one WH question per fact, in order. The model never generates the answer — we already have it (`fact["value"]`).

In [ ]:
PROMPT_TEMPLATE = """তুমি একজন বাংলা ভাষা বিশেষজ্ঞ। নিচে একটি প্যারাগ্রাফ এবং তার সাথে সম্পর্কিত কিছু তথ্য (subject, attribute, value আকারে) দেওয়া আছে।

প্রতিটি তথ্যের জন্য ঠিক একটি করে WH (কী/কে/কবে/কোথায়/কেন/কীভাবে/কোনটি/কতজন ইত্যাদি) প্রশ্ন তৈরি করো, যেন তার উত্তর হয় ঠিক সেই তথ্যের "value" এর সমান বা তার খুব কাছাকাছি।

নিয়মাবলি:
1. তালিকার প্রতিটি তথ্যের জন্য ঠিক একটি প্রশ্ন তৈরি করবে, ইনপুটের ক্রম অপরিবর্তিত রাখবে।
2. প্রশ্নটি অবশ্যই প্যারাগ্রাফের প্রসঙ্গ ও subject/attribute অনুযায়ী স্বাভাবিক বাংলায় লিখতে হবে।
3. প্রশ্নে value নিজে উল্লেখ করবে না, কারণ value-ই হলো উত্তর।
4. আউটপুট অবশ্যই একটি বৈধ JSON array of strings হতে হবে, যার দৈর্ঘ্য তথ্যের সংখ্যার ঠিক সমান এবং একই ক্রমে। অন্য কোনো ব্যাখ্যা বা টেক্সট থাকবে না।

আউটপুট ফরম্যাট (n টি তথ্যের জন্য n টি প্রশ্ন):
["প্রশ্ন ১", "প্রশ্ন ২", "..."]

প্যারাগ্রাফ:
\"\"\"{context}\"\"\"

তথ্যসমূহ:
{facts_block}
"""

def build_facts_block(facts):
    lines = []
    for i, f in enumerate(facts, 1):
        lines.append(f'{i}. subject: {f.get("subject","")}, attribute: {f.get("attribute","")}, value: {f.get("value","")}')
    return "\n".join(lines)

def build_prompt(record):
    return PROMPT_TEMPLATE.format(
        context=record.get("text", ""),
        facts_block=build_facts_block(record.get("facts", [])),
    )

def extract_json_array(text):
    """Pull out the first JSON array found in the model output, tolerating
    stray preamble/markdown fences despite instructions not to add them.
    Works for both array-of-strings and array-of-objects outputs."""
    text = text.strip()
    text = re.sub(r"^```(json)?", "", text)
    text = re.sub(r"```$", "", text)
    text = text.strip()

    start = text.find("[")
    end = text.rfind("]")
    if start == -1 or end == -1 or end < start:
        return None

    candidate = text[start:end + 1]
    try:
        data = json.loads(candidate)
        if isinstance(data, list):
            return data
    except json.JSONDecodeError:
        return None
    return None

## Batched generation
Each prompt now embeds a variable-length facts list, so `MAX_NEW_TOKENS` was raised vs. the previous version to give enough room for records with many facts.

In [ ]:
@torch.inference_mode()
def generate_batch(records):
    """Returns a list of raw decoded model outputs, one per record."""
    messages_list = [[{"role": "user", "content": build_prompt(r)}] for r in records]

    try:
        prompts = [
            tokenizer.apply_chat_template(
                m, tokenize=False, add_generation_prompt=True, enable_thinking=False
            )
            for m in messages_list
        ]
    except TypeError:
        # fallback if this tokenizer's chat template doesn't accept enable_thinking
        prompts = [
            tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
            for m in messages_list
        ]

    inputs = tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True, max_length=3072
    ).to(model.device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
    )

    input_len = inputs["input_ids"].shape[1]
    generated = output_ids[:, input_len:]
    return tokenizer.batch_decode(generated, skip_special_tokens=True)

## Load dataset
**Changed:** reads JSONL (one JSON object per line) instead of a single JSON array, and pulls `text` + `facts` instead of `context`. Records with no facts are skipped since there's nothing to generate questions for.

In [ ]:
records = []
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        if obj.get("facts"):
            records.append(obj)

if LIMIT:
    records = records[:LIMIT]

total_facts = sum(len(r["facts"]) for r in records)
print(f"Total records: {len(records)}")
print(f"Total facts (== max possible Q/A pairs): {total_facts}")

## Main loop (with checkpointing/resume)

**Changed:** the unit of work is now a *record* (title + text + facts), not a bare context string. For each record we ask the model for N questions (N = number of facts) and pair question `i` with `facts[i]["value"]` as the answer — the answer never comes from the model. If the model returns a different number of questions than facts, we only pair up to `min(len(questions), len(facts))` and log the mismatch, so we never guess an answer for a question that doesn't have one.

In [ ]:
all_pairs = []
start_idx = 0

if os.path.exists(PROGRESS_PATH):
    with open(PROGRESS_PATH, "r", encoding="utf-8") as f:
        prog = json.load(f)
    all_pairs = prog["pairs"]
    start_idx = prog["next_index"]
    print(f"Resuming from record {start_idx}, {len(all_pairs)} pairs already collected")

t0 = time.time()
for batch_start in range(start_idx, len(records), BATCH_SIZE):
    batch = records[batch_start: batch_start + BATCH_SIZE]
    raw_outputs = generate_batch(batch)

    for record, raw in zip(batch, raw_outputs):
        facts = record["facts"]
        questions = extract_json_array(raw)

        if not questions:
            print(f"  -> FAILED to parse output for record id={record.get(\'id\')}")
            continue

        if len(questions) != len(facts):
            print(f"  -> MISMATCH for record id={record.get(\'id\')}: "
                  f"{len(questions)} questions vs {len(facts)} facts")

        n = min(len(questions), len(facts))
        for i in range(n):
            q = str(questions[i]).strip()
            fact = facts[i]
            a = str(fact.get("value", "")).strip()
            if q and a:
                all_pairs.append({
                    "id": record.get("id"),
                    "context": record.get("text", ""),
                    "subject": fact.get("subject"),
                    "attribute": fact.get("attribute"),
                    "Question": q,
                    "Answer": a,
                })

    done = batch_start + len(batch)
    elapsed = time.time() - t0
    rate = done / elapsed if elapsed > 0 else 0
    print(f"[{done}/{len(records)}] pairs so far: {len(all_pairs)}  "
          f"({elapsed:.0f}s elapsed, {rate:.2f} records/s)")

    if done % CHECKPOINT_EVERY == 0 or done == len(records):
        with open(PROGRESS_PATH, "w", encoding="utf-8") as f:
            json.dump({"pairs": all_pairs, "next_index": done}, f, ensure_ascii=False)

## Save final output

In [ ]:
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(all_pairs, f, ensure_ascii=False, indent=2)

print(f"Done. Wrote {OUTPUT_PATH}")
print(f"Total Q&A pairs generated: {len(all_pairs)} (out of {total_facts} possible facts)")

## Sanity check

In [ ]:
for pair in all_pairs[:8]:
    print(pair)

coverage = len(all_pairs) / total_facts if total_facts else 0
print(f"\nFact coverage: {coverage:.1%} ({len(all_pairs)}/{total_facts})")